# Producer–Consumer Coordination

You will coordinate producers and consumers through a bounded queue and shut down cooperating workers.

You will separate item delivery from item processing, trace the rules for a queue operation that cannot proceed yet, and distinguish a completed report from a completed shutdown.

## Learning Goals

- Transfer an ordered finite sequence through a bounded queue and inspect results after both workers finish.
- Explain normal completion, backpressure, and the interruption response for a waiting worker.

## Why This Matters

A producer and consumer can work at different rates. A bounded queue limits how far delivery can get ahead while allowing each role to keep its own processing logic. Normal completion needs an agreed rule, and early shutdown needs cooperation from every affected worker. Without those rules, a caller may wait forever even when individual queue calls are thread-safe.

Applications use this separation when receiving requests and preparing reports at different rates. A storage limit gives the handoff a clear boundary. The caller also needs to know whether all agreed work finished or workers stopped early, because those outcomes support different decisions.

## Check Your Starting Point

Recall how a collection holds ordered values, how a generic type limits element types, and how a catch handles a checked exception. Explain why joining one worker says nothing about another worker’s completion.

In [ ]:
Your response:


<details>
<summary>Show answer</summary>

<a id="starting-point-interpretation"></a>

Generic collection types constrain elements. A catch handles a matching thrown exception. Join concerns only its target; inspect both results after both normal joins.

</details>

## Video Demonstration

Follow a finite delivery from a producer through a bounded queue to a consumer-owned report.

<video controls preload="metadata" width="960" aria-label="Java producer and consumer coordination demonstration">
<source src="media/03_producer_consumer_coordination/demo.mp4" type="video/mp4">
<track kind="captions" src="media/03_producer_consumer_coordination/captions.vtt" srclang="en" label="English">
</video>

[Read the producer and consumer coordination video transcript.](media/03_producer_consumer_coordination/transcript.md)

## Concept

### Pass work between two roles

A campus supply desk has a short list of item names to process: `pen`, `map`, and `book`. One worker delivers those names, and another collects them for a report. Each string represents one item request. We want every name to arrive once, in the order supplied, and both workers to finish before the caller prints the report.

A **producer** supplies items for another activity. A **consumer** receives those items and processes them. Our `WordDelivery` job is the producer's work; `WordCollector` is the consumer's work. A queue connects the roles so they can make progress separately without both editing the same result list.

Ownership still matters. The producer reads an array that this example leaves unchanged. Only the consumer changes its `received` list. The caller reads that list through a report method after joining the consumer. The queue handles the shared handoff, while these ownership rules keep the surrounding data predictable.

### Limit the number of waiting items

A **bounded blocking queue** is a shared collection with a fixed storage limit and operations that can wait when they cannot proceed. We use the `BlockingQueue` interface to describe the queue operations our jobs need. `ArrayBlockingQueue` is the concrete Java class that supplies those operations with a fixed capacity.

```java
BlockingQueue<String> queue = new ArrayBlockingQueue<String>(2);
```

This fragment assumes the queue imports shown in the complete program. The type argument `String` restricts this queue to string items. The constructor argument `2` is its **capacity**: at most two items can be stored in the queue at one time. It is not a limit of two items over the queue's lifetime. Removing an item makes space for another.

Both jobs receive this same queue object. Its operations coordinate concurrent access internally, so we do not write our own synchronized queue implementation. This protection applies to the queue operations; it does not automatically protect every object a program might use around the queue.

### Wait for the condition an operation needs

The **`put`** method adds an item, waiting for space if the bounded queue is full. The **`take`** method removes and returns an item, waiting if the queue is empty. These operations allow the two roles to coordinate without writing a loop that repeatedly asks whether space or an item is available.

The producer's loop is:

```java
for (String word : words) { queue.put(word); }
```

The enhanced `for` loop selects each string from `words` in order. Each call hands the selected string to the queue. If two items are already waiting in our capacity-two queue, the next put may have to wait until the consumer removes one. A full queue slowing the producer this way is called **backpressure**.

The consumer uses this statement inside its counted loop:

```java
received.add(queue.take());
```

Java first obtains the item returned by `take`, then passes that item to `received.add`. If no item is available yet, the take waits; the add has not happened. Once a producer supplies an item, the operation can continue and the consumer records it.

Either role may reach its queue operation first. An illustration with a full queue shows one possible ordering, not proof that an actual run filled the queue. The important rule is what each operation requires before it can complete.

### Preserve item order without choosing a worker schedule

**FIFO** means first in, first out. `ArrayBlockingQueue` removes items in the order they were inserted. Our single producer inserts `pen`, then `map`, then `book`, so the single consumer collects those names in that order.

The queue can be empty between handoffs, or it can hold more than one waiting item. Those timing differences do not change the ordering of this producer's items. The resulting list tells us the order of received names; it does not tell us which worker finished first or how long either worker waited.

A take removes its item from the queue. This is a handoff to one receiver, not a broadcast to every possible consumer. The queue also does not remove duplicate values: two separate requests with the same string remain two items. These properties distinguish this handoff from the set-based membership checks used earlier in the course.

### Agree on how normal work ends

Waiting operations need an ending rule as well as a handoff rule. Our **finite completion protocol** is an agreement about a known amount of work: the producer sends every array element, and the consumer receives exactly that many items.

```java
WordDelivery delivery = new WordDelivery(queue, words);
WordCollector collector = new WordCollector(queue, words.length);
```

These constructions assume `queue` and `words` already exist. The delivery job receives the array of names. The collector receives the number of elements it should take. Its loop stops after that many successful receives instead of checking whether the queue happens to be empty at one moment.

An empty queue does not by itself mean the producer is finished. The producer might be about to add the next item. Using the agreed count avoids confusing a temporary lack of items with the end of this finite task.

If the array is empty, both loop counts are zero, so neither role needs a handoff to finish. If the consumer expects more items than the producer sends, an extra take may wait indefinitely. Conversely, receiving too few can leave work uncollected and may leave a producer waiting for space. Our normal example assumes delivery completes and the counts agree; handling a failed or canceled role needs a separate shutdown decision.

### Let a waiting worker respond to interruption

A **thread interruption** is a cooperative request for a thread to respond. Calling its `interrupt` method does not forcibly terminate arbitrary code. The behavior depends on what that thread is doing and how its code handles the request.

The queue operations we use are interruptible. They can report **`InterruptedException`** when interrupted, including when a request is already pending as an operation begins. Each worker surrounds its queue loop with a `try` statement and handles that exception in this catch block:

```java
        } catch (InterruptedException problem) {
            Thread.currentThread().interrupt();
        }
```

This is the ending fragment of the `try` statement inside either job's `run` method, not a standalone statement to run by itself. `Thread.currentThread()` identifies the worker executing this catch block. Calling `interrupt()` on it restores its **interrupt status**, a flag recording the request, because reporting `InterruptedException` clears that status.

In these jobs, the catch block is followed by the end of `run`. The worker therefore leaves its work method instead of returning to the loop. Restoring the flag records the request; reaching the end of the method is what lets this worker terminate. We need both the signal handling and an actual exit path.

An interrupted consumer may have received only part of the normal list. Its termination would not prove that all requested items were processed. Later support examples check termination separately from normal successful delivery.

### Give the owner responsibility for stopping both roles

**Cooperative cancellation** combines a stop request with code that responds and an owner that verifies the workers have ended. The owner is the caller that created and manages the worker threads.

Stopping one queue role can affect the other. If a producer exits, a consumer could remain waiting for an item. If a consumer exits, a producer could remain waiting for space. The later shutdown example therefore manages an ongoing producer and consumer as a pair. Its owner requests interruption of both before waiting for either to finish:

```java
producer.interrupt();
consumer.interrupt();
producer.join();
consumer.join();
```

This fragment comes from that separate shutdown example, after both workers have been started. Sending both requests first gives each worker a chance to leave its interruptible operation. The joins then wait for their termination. Joining the first worker before sending the second request could leave the owner waiting while the other role still needs a stop request.

The ongoing example may process some items or none before the requests arrive. We will not infer an item count or a particular waiting state from it. After normal returns from both joins, the owner can check that neither worker remains alive. That is evidence of completed shutdown, not of completed business work.

Keep the two outcomes distinct as you study the complete programs: the finite delivery example checks that all agreed items were collected, while the cancellation example checks that both ongoing workers respond and terminate.

## Worked Example

A campus supply desk passes pen, map, and book item requests from one worker to another through a capacity-two queue, then reports the collected names in order.

```java
import java.util.ArrayList;
import java.util.concurrent.ArrayBlockingQueue;
import java.util.concurrent.BlockingQueue;
class WordDelivery implements Runnable {
    private BlockingQueue<String> queue;
    private String[] words;
    public WordDelivery(BlockingQueue<String> queue, String[] words) {
        this.queue = queue;
        this.words = words;
    }
    @Override
    public void run() {
        try {
            for (String word : words) { queue.put(word); }
        } catch (InterruptedException problem) {
            Thread.currentThread().interrupt();
        }
    }
}
class WordCollector implements Runnable {
    private BlockingQueue<String> queue;
    private int expectedCount;
    private ArrayList<String> received;
    public WordCollector(BlockingQueue<String> queue, int expectedCount) {
        this.queue = queue;
        this.expectedCount = expectedCount;
        this.received = new ArrayList<String>();
    }
    @Override
    public void run() {
        try {
            for (int index = 0; index < expectedCount; index++) {
                received.add(queue.take());
            }
        } catch (InterruptedException problem) {
            Thread.currentThread().interrupt();
        }
    }
    public String report() { return received.toString(); }
}
String[] words = {"pen", "map", "book"};
BlockingQueue<String> queue = new ArrayBlockingQueue<String>(2);
WordDelivery delivery = new WordDelivery(queue, words);
WordCollector collector = new WordCollector(queue, words.length);
Thread producer = new Thread(delivery);
Thread consumer = new Thread(collector);
producer.start();
consumer.start();
producer.join();
consumer.join();
System.out.println("Received: " + collector.report());
System.out.println("Queue empty: " + queue.isEmpty());
```

Expected output:

```text
Received: [pen, map, book]
Queue empty: true
```

The complete program gives WordDelivery the pen, map, book array and gives WordCollector the matching words.length count. Both jobs receive the same capacity-two queue. The producer reads the supplied names; the consumer alone builds the received list. Starting both before either join allows either role to make progress when the other reaches an operation that must wait.

Each successful put adds one name and each successful take removes the next name. With one producer and one consumer, the agreed three handoffs produce Received: [pen, map, book]. After both normal joins, Queue empty: true confirms that no item from this completed finite delivery remains buffered. This final state does not show whether the queue ever filled or either worker waited.

The catch blocks provide an exit if a queue operation reports interruption, but the expected report describes normal delivery. The separate shutdown support checks a different outcome: both ongoing roles have stopped, without a promised delivered-item count. Keep that distinction when comparing the worked result with later practice.

## Guided Practice

Use the explained program first to retrieve its reasoning, then apply the ideas to distinct completion, modification and debugging tasks.

Recall the already demonstrated result and explain how the program produces it. Identify the relevant owned or shared state and the rule that allows its final observation. Then run the complete example and preserve this response; this is retrieval, not an unseen prediction.

In [ ]:
Your response:


In [ ]:
import java.util.ArrayList;
import java.util.concurrent.ArrayBlockingQueue;
import java.util.concurrent.BlockingQueue;
class WordDelivery implements Runnable {
    private BlockingQueue<String> queue;
    private String[] words;
    public WordDelivery(BlockingQueue<String> queue, String[] words) {
        this.queue = queue;
        this.words = words;
    }
    @Override
    public void run() {
        try {
            for (String word : words) { queue.put(word); }
        } catch (InterruptedException problem) {
            Thread.currentThread().interrupt();
        }
    }
}
class WordCollector implements Runnable {
    private BlockingQueue<String> queue;
    private int expectedCount;
    private ArrayList<String> received;
    public WordCollector(BlockingQueue<String> queue, int expectedCount) {
        this.queue = queue;
        this.expectedCount = expectedCount;
        this.received = new ArrayList<String>();
    }
    @Override
    public void run() {
        try {
            for (int index = 0; index < expectedCount; index++) {
                received.add(queue.take());
            }
        } catch (InterruptedException problem) {
            Thread.currentThread().interrupt();
        }
    }
    public String report() { return received.toString(); }
}
String[] words = {"pen", "map", "book"};
BlockingQueue<String> queue = new ArrayBlockingQueue<String>(2);
WordDelivery delivery = new WordDelivery(queue, words);
WordCollector collector = new WordCollector(queue, words.length);
Thread producer = new Thread(delivery);
Thread consumer = new Thread(collector);
producer.start();
consumer.start();
producer.join();
consumer.join();
System.out.println("Received: " + collector.report());
System.out.println("Queue empty: " + queue.isEmpty());

Record the actual output from this run. Compare them with the already explained result. Identify any difference without changing your original retrieval response.

In [ ]:
Your response:


Trace one label from put to take, identify the agreed total count, and explain why a capacity bound is not a total delivery limit. Do not claim that a worker actually waited.

In [ ]:
Your response:


<details>
<summary>Show answer</summary>

<a id="worked-interpretation"></a>

One queue carries the three items in insertion order to the consumer-owned list. The agreed count establishes the normal ending rule; both joins precede the final reports. Capacity is not total delivery, and the output does not prove an actual wait.

</details>

<details id="animation-bounded_fifo_handoff" class="animation-panel" open>
<summary>Make room for the next handoff — show or hide animation</summary>
<p><img src="media/03_producer_consumer_coordination/bounded_fifo_handoff.gif" alt="A capacity-two queue holds pen and map. A third put of book waits for space in this illustration. Taking pen frees one slot; book is inserted, and the consumer receives map and book. Both joins precede the completed report." width="960" style="max-width:100%;height:auto;"></p>
</details>

This sequence illustrates one permitted ordering. A take removes the oldest queued name and makes space available for another put. The final report contains all three names in order; it does not prove that the illustrated waiting period occurred. This silent loop lasts 22 seconds. Hide the animation to remove visible motion.

[View this final state as a still image](media/03_producer_consumer_coordination/bounded_fifo_handoff_still.png).


<a id="animation-bounded_fifo_handoff-still"></a>

[View the final state as a still image](media/03_producer_consumer_coordination/bounded_fifo_handoff_still.png).


<details id="animation-finite_delivery_count" class="animation-panel" open>
<summary>Match the receive count to the delivery — show or hide animation</summary>
<p><img src="media/03_producer_consumer_coordination/finite_delivery_count.gif" alt="The producer supplies pen, map, and book. The consumer expects three receives and its own list grows from empty to all three names. Both completed joins precede Received: [pen, map, book] and Queue empty: true." width="960" style="max-width:100%;height:auto;"></p>
</details>

The consumer’s expected count matches the input array length. Queue capacity limits pending items, so reaching capacity does not end delivery. After three successful takes, the finite receiving loop ends. The progress counts in this diagram explain the loop; they are not another program field. This silent loop lasts 19 seconds. Hide the animation to remove visible motion.

[View this final state as a still image](media/03_producer_consumer_coordination/finite_delivery_count_still.png).


<a id="animation-finite_delivery_count-still"></a>

[View the final state as a still image](media/03_producer_consumer_coordination/finite_delivery_count_still.png).


### Request an interruptible receive to stop

The next complete program creates an empty queue and one worker whose job attempts a take. Its owner starts that worker, requests interruption, then waits for it to finish. Read the upcoming program before answering the prompt; the request can arrive before or during the queue operation. Predict the safe final reports without guessing which scheduling state occurred.

Predict how interruption affects the take call and which report is safe after the owner joins the worker. Do not predict its scheduling state at the moment interruption is requested.

In [ ]:
Your response:


In [ ]:
import java.util.concurrent.ArrayBlockingQueue;
import java.util.concurrent.BlockingQueue;
BlockingQueue<String> queue = new ArrayBlockingQueue<String>(1);
Runnable work = () -> {
    try {
        queue.take();
        System.out.println("Unexpected item");
    } catch (InterruptedException problem) {
        Thread.currentThread().interrupt();
        System.out.println("Stop requested: " + Thread.currentThread().isInterrupted());
    }
};
Thread worker = new Thread(work);
worker.start();
worker.interrupt();
worker.join();
System.out.println("Worker alive: " + worker.isAlive());

Record the actual report, identify the catch-and-return path, and explain why normal join proves termination without proving the earlier scheduling state.

In [ ]:
Your response:


<details>
<summary>Show answer</summary>

<a id="support-interrupt-answer-source-1"></a>

```java
import java.util.concurrent.ArrayBlockingQueue;
import java.util.concurrent.BlockingQueue;
BlockingQueue<String> queue = new ArrayBlockingQueue<String>(1);
Runnable work = () -> {
    try {
        queue.take();
        System.out.println("Unexpected item");
    } catch (InterruptedException problem) {
        Thread.currentThread().interrupt();
        System.out.println("Stop requested: " + Thread.currentThread().isInterrupted());
    }
};
Thread worker = new Thread(work);
worker.start();
worker.interrupt();
worker.join();
System.out.println("Worker alive: " + worker.isAlive());
```

Expected output:

```text
Stop requested: true
Worker alive: false
```

<a id="support-interrupt-interpretation"></a>

Interrupt requests that the pending take stop through InterruptedException. The handler restores the flag and returns. A normal join establishes termination; the output does not prove a particular pre-interrupt wait state.

<details id="animation-interruptible_take_exit" class="animation-panel" open>
<summary>Respond to an interrupted receive — show or hide animation</summary>
<p><img src="media/03_producer_consumer_coordination/interruptible_take_exit.gif" alt="The owner interrupts a worker taking from an empty queue. InterruptedException skips the Unexpected item statement. The handler restores the interrupt flag and prints Stop requested: true. The work method returns; join completes before Worker alive: false." width="960" style="max-width:100%;height:auto;"></p>
</details>

Interruption may arrive before or during the queue operation. The exception leads to the handler and clears the interrupt status. The handler restores that status, then reaches the end of the work method. Returning ends the work; restoring the flag alone does not end it. This silent loop lasts 19 seconds. Hide the animation to remove visible motion.


<a id="animation-interruptible_take_exit-still"></a>

[View the final state as a still image](media/03_producer_consumer_coordination/interruptible_take_exit_still.png).


</details>

### Stop both ongoing roles

The next program replaces a fixed delivery count with an ongoing producer and consumer. The owner starts both, requests interruption of both, then joins both. Read the complete source before writing your prediction. The goal is to establish that both owned workers terminate; no particular number of delivered items is promised.

Read the required two-role shutdown support. Explain why the owner interrupts both workers before joining either, then predict only the two alive reports. Do not predict a delivered-item count or a waiting state.

In [ ]:
Your response:


In [ ]:
import java.util.concurrent.ArrayBlockingQueue;
import java.util.concurrent.BlockingQueue;
class OngoingDelivery implements Runnable {
    private BlockingQueue<String> queue;
    public OngoingDelivery(BlockingQueue<String> queue) { this.queue = queue; }
    @Override
    public void run() {
        try {
            while (true) { queue.put("request"); }
        } catch (InterruptedException problem) {
            Thread.currentThread().interrupt();
        }
    }
}
class OngoingCollector implements Runnable {
    private BlockingQueue<String> queue;
    public OngoingCollector(BlockingQueue<String> queue) { this.queue = queue; }
    @Override
    public void run() {
        try {
            while (true) { queue.take(); }
        } catch (InterruptedException problem) {
            Thread.currentThread().interrupt();
        }
    }
}
BlockingQueue<String> queue = new ArrayBlockingQueue<String>(1);
Thread producer = new Thread(new OngoingDelivery(queue));
Thread consumer = new Thread(new OngoingCollector(queue));
producer.start();
consumer.start();
producer.interrupt();
consumer.interrupt();
producer.join();
consumer.join();
System.out.println("Producer alive: " + producer.isAlive());
System.out.println("Consumer alive: " + consumer.isAlive());


Record both actual alive reports. Trace each worker’s interruption exit and both owner joins. Explain why these reports cannot establish how many values were transferred.

In [ ]:
Your response:


<details>
<summary>Show answer</summary>

<a id="support-owner-cancellation-answer-source-1"></a>

```java
import java.util.concurrent.ArrayBlockingQueue;
import java.util.concurrent.BlockingQueue;
class OngoingDelivery implements Runnable {
    private BlockingQueue<String> queue;
    public OngoingDelivery(BlockingQueue<String> queue) { this.queue = queue; }
    @Override
    public void run() {
        try {
            while (true) { queue.put("request"); }
        } catch (InterruptedException problem) {
            Thread.currentThread().interrupt();
        }
    }
}
class OngoingCollector implements Runnable {
    private BlockingQueue<String> queue;
    public OngoingCollector(BlockingQueue<String> queue) { this.queue = queue; }
    @Override
    public void run() {
        try {
            while (true) { queue.take(); }
        } catch (InterruptedException problem) {
            Thread.currentThread().interrupt();
        }
    }
}
BlockingQueue<String> queue = new ArrayBlockingQueue<String>(1);
Thread producer = new Thread(new OngoingDelivery(queue));
Thread consumer = new Thread(new OngoingCollector(queue));
producer.start();
consumer.start();
producer.interrupt();
consumer.interrupt();
producer.join();
consumer.join();
System.out.println("Producer alive: " + producer.isAlive());
System.out.println("Consumer alive: " + consumer.isAlive());
```

Expected output:

```text
Producer alive: false
Consumer alive: false
```

<a id="support-owner-cancellation-interpretation"></a>

The owner requests interruption of both producer and consumer before joining either. Each worker exits its interruption handler after restoring its flag. Both normal joins precede the alive reports. The fixture makes no item-count, waiting-state or scheduling claim.

<details id="animation-two_role_owner_shutdown" class="animation-panel" open>
<summary>Request both stops before waiting — show or hide animation</summary>
<p><img src="media/03_producer_consumer_coordination/two_role_owner_shutdown.gif" alt="The owner starts an ongoing producer and consumer, requests interruption of both, then joins both. Each interruptible queue operation can leave through its handler, restore its flag, and return. The final report states that both workers are no longer alive." width="960" style="max-width:100%;height:auto;"></p>
</details>

Both owned roles need their own stop request before the caller waits for either one. Each worker restores its interruption flag and leaves its work method. The joined report confirms completed shutdown; it does not state how many items were delivered or which worker finished first. This silent loop lasts 22 seconds. Hide the animation to remove visible motion.


<a id="animation-two_role_owner_shutdown-still"></a>

[View the final state as a still image](media/03_producer_consumer_coordination/two_role_owner_shutdown_still.png).


</details>

### Complete the handoff operations

A campus supply desk sends the three labels pen, map, and book to a collector. One producer submits each label once; one consumer must receive exactly three labels in the same order. The shared queue can hold two labels.

This repairs the already explained worked program. Recall its result and explain the repair; this is not an unseen prediction. Complete SEND and RECEIVE with the queue operations that wait when the queue cannot yet accept or provide an item. Explain what each operation waits for and why both workers start before either join. Record the expected received list and final queue state. Then copy the complete repaired program into the Java work cell and run it.

Allowed changes:

- Replace SEND with the appropriate insertion operation.
- Replace RECEIVE with the appropriate removal operation.

Read the supplied source below. Keep the faulty or incomplete version as a reading example; run only your complete repaired program in the Java editing cell.

```java
import java.util.ArrayList;
import java.util.concurrent.ArrayBlockingQueue;
import java.util.concurrent.BlockingQueue;
class WordDelivery implements Runnable {
    private BlockingQueue<String> queue;
    private String[] words;
    public WordDelivery(BlockingQueue<String> queue, String[] words) {
        this.queue = queue;
        this.words = words;
    }
    @Override
    public void run() {
        try {
            for (String word : words) { queue.SEND(word); }
        } catch (InterruptedException problem) {
            Thread.currentThread().interrupt();
        }
    }
}
class WordCollector implements Runnable {
    private BlockingQueue<String> queue;
    private int expectedCount;
    private ArrayList<String> received;
    public WordCollector(BlockingQueue<String> queue, int expectedCount) {
        this.queue = queue;
        this.expectedCount = expectedCount;
        this.received = new ArrayList<String>();
    }
    @Override
    public void run() {
        try {
            for (int index = 0; index < expectedCount; index++) {
                received.add(queue.RECEIVE());
            }
        } catch (InterruptedException problem) {
            Thread.currentThread().interrupt();
        }
    }
    public String report() { return received.toString(); }
}
String[] words = {"pen", "map", "book"};
BlockingQueue<String> queue = new ArrayBlockingQueue<String>(2);
WordDelivery delivery = new WordDelivery(queue, words);
WordCollector collector = new WordCollector(queue, words.length);
Thread producer = new Thread(delivery);
Thread consumer = new Thread(collector);
producer.start();
consumer.start();
producer.join();
consumer.join();
System.out.println("Received: " + collector.report());
System.out.println("Queue empty: " + queue.isEmpty());
```

In [ ]:
Your response:


Record both actual output lines, compare with the stated expected result, and explain why queue capacity two does not limit the total delivery to two labels.

In [ ]:
Your response:


<details>
<summary>Show answer</summary>

<a id="13-03-guided-completion-answer-source-1"></a>

```java
import java.util.ArrayList;
import java.util.concurrent.ArrayBlockingQueue;
import java.util.concurrent.BlockingQueue;
class WordDelivery implements Runnable {
    private BlockingQueue<String> queue;
    private String[] words;
    public WordDelivery(BlockingQueue<String> queue, String[] words) {
        this.queue = queue;
        this.words = words;
    }
    @Override
    public void run() {
        try {
            for (String word : words) { queue.put(word); }
        } catch (InterruptedException problem) {
            Thread.currentThread().interrupt();
        }
    }
}
class WordCollector implements Runnable {
    private BlockingQueue<String> queue;
    private int expectedCount;
    private ArrayList<String> received;
    public WordCollector(BlockingQueue<String> queue, int expectedCount) {
        this.queue = queue;
        this.expectedCount = expectedCount;
        this.received = new ArrayList<String>();
    }
    @Override
    public void run() {
        try {
            for (int index = 0; index < expectedCount; index++) {
                received.add(queue.take());
            }
        } catch (InterruptedException problem) {
            Thread.currentThread().interrupt();
        }
    }
    public String report() { return received.toString(); }
}
String[] words = {"pen", "map", "book"};
BlockingQueue<String> queue = new ArrayBlockingQueue<String>(2);
WordDelivery delivery = new WordDelivery(queue, words);
WordCollector collector = new WordCollector(queue, words.length);
Thread producer = new Thread(delivery);
Thread consumer = new Thread(collector);
producer.start();
consumer.start();
producer.join();
consumer.join();
System.out.println("Received: " + collector.report());
System.out.println("Queue empty: " + queue.isEmpty());
```

Expected output:

```text
Received: [pen, map, book]
Queue empty: true
```

<a id="13-03-guided-completion-interpretation"></a>

Use put to deliver each label and take to remove the next label. The queue preserves insertion order for this one producer. The collector performs words.length removals, matching the number of puts. Both workers can make progress because both start before the caller waits for completion. After both normal joins, the report contains all three labels and the queue is empty. A successful run does not prove that either worker actually had to wait.

</details>

### Change the delivery inputs

The supply desk now scans map twice and pen once. Repeated labels represent separate scanned items and must not be removed as duplicates. The queue has room for one pending label.

Starting with the complete word-delivery program, change only the words array to map, map, pen and the queue capacity to one. Before editing, predict the received list and final queue state. Explain whether capacity changes the number of labels delivered and whether the output can establish that a wait occurred. Then make the edits and run the whole program.

Allowed changes:

- Replace only the three words array elements.
- Change queue capacity from two to one.

Read this complete baseline before writing your prediction; run your changed program in the Java editing cell.

```java
import java.util.ArrayList;
import java.util.concurrent.ArrayBlockingQueue;
import java.util.concurrent.BlockingQueue;
class WordDelivery implements Runnable {
    private BlockingQueue<String> queue;
    private String[] words;
    public WordDelivery(BlockingQueue<String> queue, String[] words) {
        this.queue = queue;
        this.words = words;
    }
    @Override
    public void run() {
        try {
            for (String word : words) { queue.put(word); }
        } catch (InterruptedException problem) {
            Thread.currentThread().interrupt();
        }
    }
}
class WordCollector implements Runnable {
    private BlockingQueue<String> queue;
    private int expectedCount;
    private ArrayList<String> received;
    public WordCollector(BlockingQueue<String> queue, int expectedCount) {
        this.queue = queue;
        this.expectedCount = expectedCount;
        this.received = new ArrayList<String>();
    }
    @Override
    public void run() {
        try {
            for (int index = 0; index < expectedCount; index++) {
                received.add(queue.take());
            }
        } catch (InterruptedException problem) {
            Thread.currentThread().interrupt();
        }
    }
    public String report() { return received.toString(); }
}
String[] words = {"pen", "map", "book"};
BlockingQueue<String> queue = new ArrayBlockingQueue<String>(2);
WordDelivery delivery = new WordDelivery(queue, words);
WordCollector collector = new WordCollector(queue, words.length);
Thread producer = new Thread(delivery);
Thread consumer = new Thread(collector);
producer.start();
consumer.start();
producer.join();
consumer.join();
System.out.println("Received: " + collector.report());
System.out.println("Queue empty: " + queue.isEmpty());
```

In [ ]:
Your response:


In [ ]:
import java.util.ArrayList;
import java.util.concurrent.ArrayBlockingQueue;
import java.util.concurrent.BlockingQueue;
class WordDelivery implements Runnable {
    private BlockingQueue<String> queue;
    private String[] words;
    public WordDelivery(BlockingQueue<String> queue, String[] words) {
        this.queue = queue;
        this.words = words;
    }
    @Override
    public void run() {
        try {
            for (String word : words) { queue.put(word); }
        } catch (InterruptedException problem) {
            Thread.currentThread().interrupt();
        }
    }
}
class WordCollector implements Runnable {
    private BlockingQueue<String> queue;
    private int expectedCount;
    private ArrayList<String> received;
    public WordCollector(BlockingQueue<String> queue, int expectedCount) {
        this.queue = queue;
        this.expectedCount = expectedCount;
        this.received = new ArrayList<String>();
    }
    @Override
    public void run() {
        try {
            for (int index = 0; index < expectedCount; index++) {
                received.add(queue.take());
            }
        } catch (InterruptedException problem) {
            Thread.currentThread().interrupt();
        }
    }
    public String report() { return received.toString(); }
}
String[] words = {"pen", "map", "book"};
BlockingQueue<String> queue = new ArrayBlockingQueue<String>(2);
WordDelivery delivery = new WordDelivery(queue, words);
WordCollector collector = new WordCollector(queue, words.length);
Thread producer = new Thread(delivery);
Thread consumer = new Thread(collector);
producer.start();
consumer.start();
producer.join();
consumer.join();
System.out.println("Received: " + collector.report());
System.out.println("Queue empty: " + queue.isEmpty());

Record both lines and compare them with your prediction. Explain how duplicate labels survive delivery. 

In [ ]:
Your response:


After recording the changed case, restore the already taught baseline inputs and run a fresh complete program in the next cell.

Record the actual restored baseline result and compare it with the earlier worked result. Explain which input you restored and why this run creates fresh work objects.

In [ ]:
Your response:


<details>
<summary>Show answer</summary>

<a id="13-03-guided-modification-answer-source-1"></a>

```java
import java.util.ArrayList;
import java.util.concurrent.ArrayBlockingQueue;
import java.util.concurrent.BlockingQueue;
class WordDelivery implements Runnable {
    private BlockingQueue<String> queue;
    private String[] words;
    public WordDelivery(BlockingQueue<String> queue, String[] words) {
        this.queue = queue;
        this.words = words;
    }
    @Override
    public void run() {
        try {
            for (String word : words) { queue.put(word); }
        } catch (InterruptedException problem) {
            Thread.currentThread().interrupt();
        }
    }
}
class WordCollector implements Runnable {
    private BlockingQueue<String> queue;
    private int expectedCount;
    private ArrayList<String> received;
    public WordCollector(BlockingQueue<String> queue, int expectedCount) {
        this.queue = queue;
        this.expectedCount = expectedCount;
        this.received = new ArrayList<String>();
    }
    @Override
    public void run() {
        try {
            for (int index = 0; index < expectedCount; index++) {
                received.add(queue.take());
            }
        } catch (InterruptedException problem) {
            Thread.currentThread().interrupt();
        }
    }
    public String report() { return received.toString(); }
}
String[] words = {"map", "map", "pen"};
BlockingQueue<String> queue = new ArrayBlockingQueue<String>(1);
WordDelivery delivery = new WordDelivery(queue, words);
WordCollector collector = new WordCollector(queue, words.length);
Thread producer = new Thread(delivery);
Thread consumer = new Thread(collector);
producer.start();
consumer.start();
producer.join();
consumer.join();
System.out.println("Received: " + collector.report());
System.out.println("Queue empty: " + queue.isEmpty());
```

Expected output:

```text
Received: [map, map, pen]
Queue empty: true
```

<a id="13-03-guided-modification-interpretation"></a>

The queue delivers both map entries because it does not remove duplicates. Capacity one bounds the number of pending items, not the total number delivered. The single producer supplies map, map, pen in that order and the consumer removes each once. Both joins complete before the caller reads the collected list. No exact wait duration or worker schedule follows from these two output lines.

</details>

### Repair the completion count

A maintainer changes the consumer count while leaving the three delivered labels unchanged. The program is intended to receive every delivered label once and then return control to the caller.

This repairs the already explained worked program. Recall its result and explain the repair; this is not an unseen prediction. Read this faulty program without running it. The collector now expects words.length + 1 items. Trace what happens after the three actual labels are taken. Identify the operation that can wait for an item no producer will supply, and explain why the caller cannot rely on reaching the final print statements. Correct the count while preserving the three labels and both worker bodies. Predict the repaired result, then place only the complete repaired program in the Java work cell and run it.

Allowed changes:

- Replace words.length + 1 with the count matching actual deliveries.

Read the supplied source below. Keep the faulty or incomplete version as a reading example; run only your complete repaired program in the Java editing cell.

```java
import java.util.ArrayList;
import java.util.concurrent.ArrayBlockingQueue;
import java.util.concurrent.BlockingQueue;
class WordDelivery implements Runnable {
    private BlockingQueue<String> queue;
    private String[] words;
    public WordDelivery(BlockingQueue<String> queue, String[] words) {
        this.queue = queue;
        this.words = words;
    }
    @Override
    public void run() {
        try {
            for (String word : words) { queue.put(word); }
        } catch (InterruptedException problem) {
            Thread.currentThread().interrupt();
        }
    }
}
class WordCollector implements Runnable {
    private BlockingQueue<String> queue;
    private int expectedCount;
    private ArrayList<String> received;
    public WordCollector(BlockingQueue<String> queue, int expectedCount) {
        this.queue = queue;
        this.expectedCount = expectedCount;
        this.received = new ArrayList<String>();
    }
    @Override
    public void run() {
        try {
            for (int index = 0; index < expectedCount; index++) {
                received.add(queue.take());
            }
        } catch (InterruptedException problem) {
            Thread.currentThread().interrupt();
        }
    }
    public String report() { return received.toString(); }
}
String[] words = {"pen", "map", "book"};
BlockingQueue<String> queue = new ArrayBlockingQueue<String>(2);
WordDelivery delivery = new WordDelivery(queue, words);
WordCollector collector = new WordCollector(queue, words.length + 1);
Thread producer = new Thread(delivery);
Thread consumer = new Thread(collector);
producer.start();
consumer.start();
producer.join();
consumer.join();
System.out.println("Received: " + collector.report());
System.out.println("Queue empty: " + queue.isEmpty());
```

In [ ]:
Your response:


Record the repaired output and explain why matching counts allows this finite example to finish. Distinguish the repaired program you ran from the faulty trace you only read.

In [ ]:
Your response:


<details>
<summary>Show answer</summary>

<a id="13-03-guided-debugging-answer-source-1"></a>

```java
import java.util.ArrayList;
import java.util.concurrent.ArrayBlockingQueue;
import java.util.concurrent.BlockingQueue;
class WordDelivery implements Runnable {
    private BlockingQueue<String> queue;
    private String[] words;
    public WordDelivery(BlockingQueue<String> queue, String[] words) {
        this.queue = queue;
        this.words = words;
    }
    @Override
    public void run() {
        try {
            for (String word : words) { queue.put(word); }
        } catch (InterruptedException problem) {
            Thread.currentThread().interrupt();
        }
    }
}
class WordCollector implements Runnable {
    private BlockingQueue<String> queue;
    private int expectedCount;
    private ArrayList<String> received;
    public WordCollector(BlockingQueue<String> queue, int expectedCount) {
        this.queue = queue;
        this.expectedCount = expectedCount;
        this.received = new ArrayList<String>();
    }
    @Override
    public void run() {
        try {
            for (int index = 0; index < expectedCount; index++) {
                received.add(queue.take());
            }
        } catch (InterruptedException problem) {
            Thread.currentThread().interrupt();
        }
    }
    public String report() { return received.toString(); }
}
String[] words = {"pen", "map", "book"};
BlockingQueue<String> queue = new ArrayBlockingQueue<String>(2);
WordDelivery delivery = new WordDelivery(queue, words);
WordCollector collector = new WordCollector(queue, words.length);
Thread producer = new Thread(delivery);
Thread consumer = new Thread(collector);
producer.start();
consumer.start();
producer.join();
consumer.join();
System.out.println("Received: " + collector.report());
System.out.println("Queue empty: " + queue.isEmpty());
```

Expected output:

```text
Received: [pen, map, book]
Queue empty: true
```

<a id="13-03-guided-debugging-interpretation"></a>

After receiving three labels, the faulty collector attempts a fourth take from an empty queue. The producer has no fourth label to send, so the collector may remain waiting and consumer.join cannot establish completion. Use words.length so required removals equal actual deliveries. This finite protocol assumes the producer completes its agreed delivery; it does not recover from arbitrary producer failure. The separate owner-cancellation example teaches how an owner requests both ongoing roles to stop.

</details>

## Independent Practice

Build a complete program that transfers the taught mechanism to the following task. Keep the explained answer closed while planning, implementing and testing.

A supply audit sends integer values 2, 3 and 4 from one NumberDelivery producer to one TotalCollector consumer through a BlockingQueue<Integer> with capacity 1. Write both Runnable classes. The producer puts each value once; the consumer takes exactly values.length items and owns two int fields, both initially zero: received counts the values read, and total stores their sum. Each run catches InterruptedException, restores its current thread’s flag and returns. Start both before joining either, then print Received, Total and Queue empty. Plan the count agreement and predict all three reports.

In [ ]:
Your response:


Record every actual baseline report. Compare it with your plan, explain the mechanism, and retain any discrepancy as evidence for a repair.

In [ ]:
Your response:


Predict Received, Total and Queue empty for each fresh program: empty values with capacity 1; values 0,3,3 with capacity 1; original values 2,3,4 with capacity 2. Preserve the producer/consumer classes and set expectedCount to the actual array length. Explain why capacity changes permitted buffering rather than the required delivery count.

In [ ]:
Your response:


Record the actual result for each named variant separately, preserving the predictions. Explain what boundary each checks and what it cannot establish about scheduling.

In [ ]:
Your response:


<details>
<summary>Show answer</summary>

<a id="independent-answer-source-1"></a>

### Baseline: three values

The collector receives three items and adds 2, 3, and 4 to obtain nine. Both workers finish their agreed count, so the completed queue contains no pending item.

```java
import java.util.concurrent.ArrayBlockingQueue;
import java.util.concurrent.BlockingQueue;
class NumberDelivery implements Runnable {
    private BlockingQueue<Integer> queue;
    private int[] values;
    public NumberDelivery(BlockingQueue<Integer> queue, int[] values) {
        this.queue = queue;
        this.values = values;
    }
    @Override
    public void run() {
        try {
            for (int value : values) { queue.put(value); }
        } catch (InterruptedException problem) {
            Thread.currentThread().interrupt();
        }
    }
}
class TotalCollector implements Runnable {
    private BlockingQueue<Integer> queue;
    private int expectedCount;
    private int total;
    private int received;
    public TotalCollector(BlockingQueue<Integer> queue, int expectedCount) {
        this.queue = queue;
        this.expectedCount = expectedCount;
        this.total = 0;
        this.received = 0;
    }
    @Override
    public void run() {
        try {
            for (int index = 0; index < expectedCount; index++) {
                total += queue.take();
                received++;
            }
        } catch (InterruptedException problem) {
            Thread.currentThread().interrupt();
        }
    }
    public int getTotal() { return total; }
    public int getReceived() { return received; }
}
int[] values = {2, 3, 4};
BlockingQueue<Integer> queue = new ArrayBlockingQueue<Integer>(1);
NumberDelivery delivery = new NumberDelivery(queue, values);
TotalCollector collector = new TotalCollector(queue, values.length);
Thread producer = new Thread(delivery);
Thread consumer = new Thread(collector);
producer.start();
consumer.start();
producer.join();
consumer.join();
System.out.println("Received: " + collector.getReceived());
System.out.println("Total: " + collector.getTotal());
System.out.println("Queue empty: " + queue.isEmpty());
```

Expected output:

```text
Received: 3
Total: 9
Queue empty: true
```

<a id="independent-answer-source-2"></a>

### Empty input

An empty array requires zero puts and zero takes. Both workers finish without a handoff; the count and total stay zero, and the queue is empty.

```java
import java.util.concurrent.ArrayBlockingQueue;
import java.util.concurrent.BlockingQueue;
class NumberDelivery implements Runnable {
    private BlockingQueue<Integer> queue;
    private int[] values;
    public NumberDelivery(BlockingQueue<Integer> queue, int[] values) {
        this.queue = queue;
        this.values = values;
    }
    @Override
    public void run() {
        try {
            for (int value : values) { queue.put(value); }
        } catch (InterruptedException problem) {
            Thread.currentThread().interrupt();
        }
    }
}
class TotalCollector implements Runnable {
    private BlockingQueue<Integer> queue;
    private int expectedCount;
    private int total;
    private int received;
    public TotalCollector(BlockingQueue<Integer> queue, int expectedCount) {
        this.queue = queue;
        this.expectedCount = expectedCount;
        this.total = 0;
        this.received = 0;
    }
    @Override
    public void run() {
        try {
            for (int index = 0; index < expectedCount; index++) {
                total += queue.take();
                received++;
            }
        } catch (InterruptedException problem) {
            Thread.currentThread().interrupt();
        }
    }
    public int getTotal() { return total; }
    public int getReceived() { return received; }
}
int[] values = {};
BlockingQueue<Integer> queue = new ArrayBlockingQueue<Integer>(1);
NumberDelivery delivery = new NumberDelivery(queue, values);
TotalCollector collector = new TotalCollector(queue, values.length);
Thread producer = new Thread(delivery);
Thread consumer = new Thread(collector);
producer.start();
consumer.start();
producer.join();
consumer.join();
System.out.println("Received: " + collector.getReceived());
System.out.println("Total: " + collector.getTotal());
System.out.println("Queue empty: " + queue.isEmpty());
```

Expected output:

```text
Received: 0
Total: 0
Queue empty: true
```

<a id="independent-answer-source-3"></a>

### Zero and repeated values

Zero is an ordinary item and counts as one receive. The two occurrences of three also remain separate items. Three receives produce a total of six.

```java
import java.util.concurrent.ArrayBlockingQueue;
import java.util.concurrent.BlockingQueue;
class NumberDelivery implements Runnable {
    private BlockingQueue<Integer> queue;
    private int[] values;
    public NumberDelivery(BlockingQueue<Integer> queue, int[] values) {
        this.queue = queue;
        this.values = values;
    }
    @Override
    public void run() {
        try {
            for (int value : values) { queue.put(value); }
        } catch (InterruptedException problem) {
            Thread.currentThread().interrupt();
        }
    }
}
class TotalCollector implements Runnable {
    private BlockingQueue<Integer> queue;
    private int expectedCount;
    private int total;
    private int received;
    public TotalCollector(BlockingQueue<Integer> queue, int expectedCount) {
        this.queue = queue;
        this.expectedCount = expectedCount;
        this.total = 0;
        this.received = 0;
    }
    @Override
    public void run() {
        try {
            for (int index = 0; index < expectedCount; index++) {
                total += queue.take();
                received++;
            }
        } catch (InterruptedException problem) {
            Thread.currentThread().interrupt();
        }
    }
    public int getTotal() { return total; }
    public int getReceived() { return received; }
}
int[] values = {0, 3, 3};
BlockingQueue<Integer> queue = new ArrayBlockingQueue<Integer>(1);
NumberDelivery delivery = new NumberDelivery(queue, values);
TotalCollector collector = new TotalCollector(queue, values.length);
Thread producer = new Thread(delivery);
Thread consumer = new Thread(collector);
producer.start();
consumer.start();
producer.join();
consumer.join();
System.out.println("Received: " + collector.getReceived());
System.out.println("Total: " + collector.getTotal());
System.out.println("Queue empty: " + queue.isEmpty());
```

Expected output:

```text
Received: 3
Total: 6
Queue empty: true
```

<a id="independent-answer-source-4"></a>

### More buffering space

Capacity two permits an additional pending item. The original three values still require three handoffs and sum to nine. The final report does not reveal whether either worker waited.

```java
import java.util.concurrent.ArrayBlockingQueue;
import java.util.concurrent.BlockingQueue;
class NumberDelivery implements Runnable {
    private BlockingQueue<Integer> queue;
    private int[] values;
    public NumberDelivery(BlockingQueue<Integer> queue, int[] values) {
        this.queue = queue;
        this.values = values;
    }
    @Override
    public void run() {
        try {
            for (int value : values) { queue.put(value); }
        } catch (InterruptedException problem) {
            Thread.currentThread().interrupt();
        }
    }
}
class TotalCollector implements Runnable {
    private BlockingQueue<Integer> queue;
    private int expectedCount;
    private int total;
    private int received;
    public TotalCollector(BlockingQueue<Integer> queue, int expectedCount) {
        this.queue = queue;
        this.expectedCount = expectedCount;
        this.total = 0;
        this.received = 0;
    }
    @Override
    public void run() {
        try {
            for (int index = 0; index < expectedCount; index++) {
                total += queue.take();
                received++;
            }
        } catch (InterruptedException problem) {
            Thread.currentThread().interrupt();
        }
    }
    public int getTotal() { return total; }
    public int getReceived() { return received; }
}
int[] values = {2, 3, 4};
BlockingQueue<Integer> queue = new ArrayBlockingQueue<Integer>(2);
NumberDelivery delivery = new NumberDelivery(queue, values);
TotalCollector collector = new TotalCollector(queue, values.length);
Thread producer = new Thread(delivery);
Thread consumer = new Thread(collector);
producer.start();
consumer.start();
producer.join();
consumer.join();
System.out.println("Received: " + collector.getReceived());
System.out.println("Total: " + collector.getTotal());
System.out.println("Queue empty: " + queue.isEmpty());
```

Expected output:

```text
Received: 3
Total: 9
Queue empty: true
```

<a id="independent-interpretation"></a>

The producer puts each value once, and the collector takes the agreed count. Empty input requires zero puts and takes; duplicate and zero values are data, not termination signals. Capacity bounds buffered elements, not the total delivered count. Each normal join establishes completion before the caller reads the collector’s fields.

</details>

## Summary

From memory, explain these lesson ideas in a connected account: Producer-consumer roles, Bounded blocking queue, Blocking coordination, FIFO handoff, Finite completion protocol, Thread interruption, Cooperative cancellation. Include one limit of the example evidence.

In [ ]:
Your response:


<details>
<summary>Show answer</summary>

<a id="summary-retrieval-interpretation"></a>

Producer and consumer cooperate through a bounded queue. put and take can wait for space or an item; FIFO preserves insertion order here. Capacity limits stored items, while the agreed item count controls normal completion. On interruption, these workers restore the flag and exit. The owner requests both stops before joining either, and completed shutdown is different from completed delivery.

</details>

## Reflection

Describe a campus or project task that could use this lesson’s mechanism. Identify the work, owned or shared state, completion rule and one limitation of the analogy. Explain what would fail if the rule were omitted.

In [ ]:
Your response:


The next lesson uses background work with a JavaFX interface. A waiting worker must not prevent the UI thread from handling input.

## Supplemental Reading

- [Java 21 BlockingQueue](https://docs.oracle.com/en/java/javase/21/docs/api/java.base/java/util/concurrent/BlockingQueue.html) explains the waiting insertion and removal operations.
- [Java 21 ArrayBlockingQueue](https://docs.oracle.com/en/java/javase/21/docs/api/java.base/java/util/concurrent/ArrayBlockingQueue.html) documents fixed capacity and first-in, first-out ordering.
- [Java 21 Thread](https://docs.oracle.com/en/java/javase/21/docs/api/java.base/java/lang/Thread.html) documents interruption, currentThread, join, and isAlive.
- [Java 21 InterruptedException](https://docs.oracle.com/en/java/javase/21/docs/api/java.base/java/lang/InterruptedException.html) explains how interruption can be reported while a thread waits.
- [Java 21 concurrency utilities](https://docs.oracle.com/en/java/javase/21/docs/api/java.base/java/util/concurrent/package-summary.html) describes the library that supplies the queue operations.
- [Java 21 ArrayList](https://docs.oracle.com/en/java/javase/21/docs/api/java.base/java/util/ArrayList.html) documents the consumer-owned result list.
- [Java 21 Runnable](https://docs.oracle.com/en/java/javase/21/docs/api/java.base/java/lang/Runnable.html) describes the work method each job implements.